# Problem 8 (100 points)

Residual networks (ResNets) introduced the **skip connection** $y = F(x) + x$, enabling training of networks with 100+ layers. The key insight is that learning a residual $F(x)$ is easier than learning the full mapping, and the identity shortcut guarantees gradient flow. In this problem, you will implement ResNet basic blocks and bottleneck blocks, compare gradient flow with and without skip connections, and build a working mini-ResNet.

We use the following notation in this problem.
- $F(x)$ — residual function (the convolutional pathway inside a block).
- Identity shortcut: $y = F(x) + x$ (when input and output dimensions match).
- Projection shortcut: $y = F(x) + W_s x$ (when dimensions differ, using a $1 \times 1$ convolution).
- Basic block: two $3 \times 3$ convolutions.
- Bottleneck block: $1 \times 1 \to 3 \times 3 \to 1 \times 1$ convolutions with channel reduction.

In [ ]:
# Run code in this cell

"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)

> WARNING !!!
>
- Beyond importing libraries/modules/classes/functions in the preceding cell, you are **NOT allowed to import anything else for the following purposes**:
    - **As a part of your final solution.**
    - **Temporarily import something to assist you to get a solution.**

## Part 1 (20 points, coding task)

**Do the following tasks.**

Implement `BasicBlock` — the residual block used in ResNet-18 and ResNet-34.

Structure:
```
x -> Conv2d(3x3, stride) -> BN -> ReLU -> Conv2d(3x3) -> BN -> (+shortcut) -> ReLU -> y
```

- All convolutions use `bias=False` (batch norm handles the bias role).
- If `in_channels != out_channels` or `stride != 1`, create a **projection shortcut**: `Conv2d(1x1, stride) -> BN`.
- Otherwise, the shortcut is the identity.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        ...
    
    def forward(self, x):
        """x: (B, in_channels, H, W) -> (B, out_channels, H/stride, W/stride)"""
        ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
# Identity shortcut
block1 = BasicBlock(64, 64, stride=1)
x1 = torch.randn(2, 64, 16, 16)
y1 = block1(x1)
assert y1.shape == (2, 64, 16, 16), f"Expected (2,64,16,16), got {y1.shape}"

# Projection shortcut (downsample)
block2 = BasicBlock(64, 128, stride=2)
x2 = torch.randn(2, 64, 16, 16)
y2 = block2(x2)
assert y2.shape == (2, 128, 8, 8), f"Expected (2,128,8,8), got {y2.shape}"

# Gradient flows through skip
x3 = torch.randn(1, 64, 8, 8, requires_grad=True)
block1(x3).sum().backward()
assert x3.grad is not None and x3.grad.abs().sum() > 0

print("Part 1 passed!")

For deeper ResNets (50, 101, 152), a **bottleneck** design reduces computation by using $1 \times 1$ convolutions to shrink and expand channels.

## Part 2 (20 points, coding task)

**Do the following tasks.**

Implement `BottleneckBlock` — the residual block used in ResNet-50/101/152.

Structure:
```
x -> Conv2d(1x1, reduce) -> BN -> ReLU
  -> Conv2d(3x3, stride)  -> BN -> ReLU
  -> Conv2d(1x1, expand)  -> BN
  -> (+shortcut) -> ReLU -> y
```

- Expansion factor = 4: output channels = `mid_channels * 4`.
- The $1 \times 1$ convolutions reduce to `mid_channels`, the $3 \times 3$ operates at `mid_channels`, and the final $1 \times 1$ expands to `mid_channels * 4`.
- All convolutions use `bias=False`.
- Projection shortcut when `in_channels != mid_channels * 4` or `stride != 1`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class BottleneckBlock(nn.Module):
    expansion = 4
    
    def __init__(self, in_channels, mid_channels, stride=1):
        """
        in_channels: input channels
        mid_channels: bottleneck channels (output = mid_channels * 4)
        """
        super().__init__()
        ...
    
    def forward(self, x):
        ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
# Same dimensions
b1 = BottleneckBlock(256, 64, stride=1)
x1 = torch.randn(2, 256, 8, 8)
y1 = b1(x1)
assert y1.shape == (2, 256, 8, 8), f"Expected (2,256,8,8), got {y1.shape}"

# Downsample
b2 = BottleneckBlock(256, 128, stride=2)
y2 = b2(x1)
assert y2.shape == (2, 512, 4, 4), f"Expected (2,512,4,4), got {y2.shape}"

# First block (64 -> 256)
b3 = BottleneckBlock(64, 64, stride=1)
x3 = torch.randn(2, 64, 16, 16)
y3 = b3(x3)
assert y3.shape == (2, 256, 16, 16), f"Expected (2,256,16,16), got {y3.shape}"

print("Part 2 passed!")

The fundamental advantage of skip connections is better gradient flow. Let us measure this experimentally.

## Part 3 (20 points, coding task)

**Do the following tasks.**

**Gradient flow comparison.** Build two 20-layer networks and compare gradient norms at the first layer.

- `PlainNet20`: 20 layers of `Linear(64, 64) -> ReLU` (no skip connections), ending with `Linear(64, 1)`.
- `ResNet20`: 10 residual blocks, each containing `Linear(64, 64) -> ReLU -> Linear(64, 64)` with a skip connection `(+x) -> ReLU`, ending with `Linear(64, 1)`.

For each:
1. Forward pass with `x = torch.randn(8, 64)`.
2. Compute MSE loss against random targets.
3. Backward pass.
4. Record the L2 norm of the **first layer**'s weight gradient.

Store as `plain_grad_norm` and `res_grad_norm` (floats).

In [ ]:
### WRITE YOUR SOLUTION HERE ###

torch.manual_seed(42)

class PlainNet20(nn.Module):
    ...

class ResNet20(nn.Module):
    ...

plain_grad_norm = ...  # float
res_grad_norm = ...    # float

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
assert isinstance(plain_grad_norm, float)
assert isinstance(res_grad_norm, float)
assert res_grad_norm > plain_grad_norm, \
    f"ResNet gradient ({res_grad_norm:.6f}) should be larger than plain ({plain_grad_norm:.6f})"
print(f"Part 3 passed! Plain: {plain_grad_norm:.8f}, ResNet: {res_grad_norm:.8f}")
print(f"Ratio: {res_grad_norm / max(plain_grad_norm, 1e-10):.1f}x")

Let us now assemble a complete mini-ResNet for CIFAR-10.

## Part 4 (15 points, coding task)

**Do the following tasks.**

Build `MiniResNet` for CIFAR-10 using `BasicBlock` from Part 1.

Architecture:
```
Conv2d(3, 16, 3, pad=1, bias=False) -> BN(16) -> ReLU ->
BasicBlock(16, 16)  ->
BasicBlock(16, 32, stride=2) ->
BasicBlock(32, 64, stride=2) ->
AdaptiveAvgPool2d(1) -> Flatten -> Linear(64, 10)
```

Store as `mini_resnet`. Count and store total parameters as `total_params`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class MiniResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        ...
    
    def forward(self, x):
        """x: (B, 3, 32, 32) -> (B, 10)"""
        ...

mini_resnet = MiniResNet()
total_params = sum(p.numel() for p in mini_resnet.parameters())

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
x = torch.randn(4, 3, 32, 32)
logits = mini_resnet(x)
assert logits.shape == (4, 10), f"Expected (4, 10), got {logits.shape}"
assert total_params > 0
print(f"Part 4 passed! MiniResNet parameters: {total_params:,}")

Let us consolidate our understanding with conceptual questions.

## Part 5 (10 points, non-coding task)

**Do the following tasks (Reasoning is required).**

1. The ResNet paper shows that a 56-layer **plain** network has higher **training error** than a 20-layer plain network. This is NOT overfitting (it is not a generalization gap). What causes this degradation, and how do skip connections solve it?

2. The gradient of a residual block is $\frac{\partial}{\partial x}[F(x) + x] = F'(x) + I$. In a chain of $L$ residual blocks, the gradient from the loss to the input can be expanded as a sum of $2^L$ terms (each path either goes through or skips each block). Explain why this means the gradient includes a term that does **not** decay with depth.

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 6 (15 points, non-coding task)

**Do the following tasks (Reasoning is required).**

1. A basic block with two `Conv2d(C, C, 3, pad=1, bias=False)` + two `BN(C)` layers has how many parameters? A bottleneck block with `Conv2d(4C, C, 1, bias=False)` + `BN(C)` + `Conv2d(C, C, 3, pad=1, bias=False)` + `BN(C)` + `Conv2d(C, 4C, 1, bias=False)` + `BN(4C)` has how many? Compute for $C = 64$ and compare.

2. Pre-activation ResNet places BN and ReLU **before** the convolution: `BN -> ReLU -> Conv` instead of `Conv -> BN -> ReLU`. What advantage does this have for gradient flow through the skip connection?

3. ResNeXt replaces the $3 \times 3$ convolution in a bottleneck with a **grouped convolution** `Conv2d(C, C, 3, groups=32)`. For $C = 128$, compute the parameter count of a standard $3 \times 3$ conv vs. the grouped version. What is the reduction factor?

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """